I don't really know where to start with the optimization so I'm just going to do parts here and there and then we can slowly piece it all together. I will do my best to explain the logic behind the function and what they mean as I understand them. In order to understand them I have been using ChatGPT but I am going through the text to check if what chat says makes sense in the context of the paper and I think it does most of the time. The implementations of the functions are still all my own however and I will try to explain how they work and how I understand them

First I'll just get the matrixes that they use in the numerical example just so that there is a visualization of what is being used.

In [2]:
import numpy as np

In [6]:
q_11 = np.matrix('-17 8 0 5 4 0; 6 -30 11 6 1 6; 0 7 -10 0 1 2; 6 5 4 -22 2 5; 6 9 8 3 -32 6; 7 5 2 0 1 -15').A

q_12 = np.matrix('-20 2 9 0 8 1; 4 -30 0 4 9 13; 1 5 -25 9 9 1; 4 6 1 -39 8 20; 8 4 1 7 -29 9; 4 2 6 0 5 -17').A

q_21 = np.matrix('-22 9 1 7 4 1; 1 -20 9 3 6 1; 1 6 -9 2 0 0; 7 7 2 -19 3 0; 2 4 7 6 -23 4; 6 6 4 4 6 -26').A

q_22 = np.matrix('-15 0 2 2 1 0; 0 -49 5 30 6 8; 1 1 -24 9 9 4; 6 1 7 -24 1 9; 18 3 5 5 -35 4; 11 15 1 3 3 -33').A

d_1 = np.matrix('0.0006, 0.9994; 0.5307, 0.4693; 0.4568, 0.5432; 0.7175, 0.2825; 0.6003, 0.3997; 0.4366, 0.5634').A

d_2 = np.matrix('0.4880 0.5120; 0.7214 0.2786; 0.4724 0.5276; 0.4379 0.5621; 0.6878 0.3122; 0.6511 0.3489').A

In [4]:
def create_prob_matrix(transition_rates):
    prob_matrix = np.zeros(transition_rates.shape)
    for idx in range(len(transition_rates)):
        sum = 0
        for transition in transition_rates[idx]:
            sum += transition if transition > 0 else 0
        for idx2, transition in enumerate(transition_rates[idx]):
            prob_matrix[idx][idx2] = transition/sum if transition > 0 else 0
    return prob_matrix

In [ ]:
pi_11 = create_prob_matrix(q_11)

pi_12 = create_prob_matrix(q_12)

pi_21 = create_prob_matrix(q_21)

pi_22 = create_prob_matrix(q_22)

pi_player_1 = np.array([pi_11, pi_12])

pi_player_2 = np.array([pi_21, pi_22])

pi = np.array([pi_player_1, pi_player_2])

First is the W function that can be found in equation 28. I'm doing this first just cos I feel like I can, no more reason than that. 

How I understand the W function, it is essentially the expected cost after one transition. As arguments it takes the current state and the action that each player is at and will compute the expected value coming from those states and actions. In order to do that, it will take every single possible outcome from those actions and for each outcome calculate the cost of that outcome and the probability of that outcome, multiply them together and then sum up the products of every outcome. As you might expect, this can get very big, very quickly, especially if the amount of players goes up, then it'll go up exponentially. I'm just going to be doing it for 2 players for the moment, then you can still calculate the exact value of W. With more players, apparently it's better to do an approximation of it since otherwise it gets too heavy but I don't know how to do that and will only look at that later. For the cost of the outcome that I was talking about, that means V, I don't think it's exactly the same as the one that is talked about in equation 28 since otherwise we go in circles. But apparently, which makes sense to me, V is a function that we choose ourselves, the example that chatGPT gave me is V = 1 if the capture condition is met, otherwise it equals 0 (this is for the defenders and could be the opposite for the attackers). This is a very simple one but it gives you the idea. I don't know if they have any that they specifically use in the paper. They talk about cost functions in the numerical example but that uses W, excpet I use V to calculate W so it goes in circles and I don't know what to do with that.
In the implementation, I'm just gonna call another function and not implement that since that part confuses me.

In [ ]:
def cost_function_V(player, state_player_1, action_player_1, new_state_player_1, state_player_2, action_player_2, new_state_player_2):
    '''
    I'm not implementing this, this is the function that I don't know what to do with, but these are the arguments that it would take according to the paper, I will be calling this
    This function also differs per player
    '''
    #if you want to test the expected_cost_W function you can just make this function return 1
    # return 1
    pass

In [13]:
def expected_cost_W(player, state_player_1, action_player_1, state_player_2, action_player_2):
    '''
    This is assuming that the state is less than 6 (0-5) and the action is 0 or 1.
    This function also differs per player
    '''
    W = 0
    # for loop for player 1, looping over all states
    for j1 in range(6):
        # for loop for player 2, looping over all states
        for j2 in range(6):
            cost = cost_function_V(player, state_player_1, action_player_1, j1, state_player_2, action_player_2, j2)
            probability = 1
            # probability of player 1 being in state j1
            probability *= pi_player_1[action_player_1][state_player_1][j1]
            # probability of player 2 being in state j2
            probability *= pi_player_2[action_player_2][state_player_2][j2]
            W += cost * probability
    return W

print(expected_cost_W(1, 2, 1, 4, 0))

0.9999999999999999


For the A matrix which I am doing next, the A matrix is essentially a matrix checking if the c stuff is admissable. This therefore goes over the constraints in equation 29 and 30. Equation 29 is probability constraints that if you sum up all c, then they are equal to 1. This makes sense since c(i,k) represents the probability of being in state i and doing action k, so if you were to sum them up, you would get 1. Equation 30 is checking the markov property, maybe not quite that name but something like that. And it is saying that the probability of being in a state j (sum of all actions k for state j c(j,k)  will give you the prob of being in state j) is equal to the probability of getting to state j (for all states and action, the sum of the probabilities to get to state j times the prob of being in that state and action (pi(j|i,k) * c(i,k))). So the matrix A checks these two conditions. It does that by multiplying itself by a vector that contains all the c and checks if it is equal to a vector b.

To construct the matrix A for one single player, you make the first line all 1, this will check the prob constraint of equation 29, and the rest is one line for every state. It is hard to explain in writing so I will try add good comments to my code so you can understand it but it is essentially pi with sometimes having pi-1 in the cells. This will hopefully makes sense in the code, if it doesn't, ask on the whatsapp group chat and I can record a long as voice message or make a nice diagram to help it be understood.

The b is essentially a vector with only 0 except for the first line which is a 1 for the probability constraint.

Within the paper, they also talk about matrix A_eq with having BL and e in it. This is essentially a mash up of all A matrices of every player put together. BL is the matrix that checks the markov constraint and e is the vector that checks the probability constraint. I think I understand it decently well now so if you have questions about the A_eq than feel free to ask on whatsapp.
Again, the b_eq is all 0 except for the lines with e where its equal to 1. Though the paper does not help much to understand that, it looks like a random distribution of 0 and 1 but it isnt.

In [26]:
def build_A_matrix(state = 6, action = 2):
    '''
    Takes the amount of states there can be and amount of actions that can be taken, this is to make the shape of the matrix.
    This assumes that you can take every action from every state, or at least that there is a c value attributed to every combination of state and action, even if the c is equal to 0 thats fine as long as there is one
    Right now only works when state = 6 and action = 2 because of the used pi matrices
    '''
    # checks the probability constraint which just sums up all the c so it is a 1 in every cell
    A_matrix = np.ones(shape=state*action)

    # creates a line for every state to check the markov constraint and adds it to the A_matrix
    for j in range(state):
        # I'm going to be using the pi matrices above for this example so this function will only work for 6 states and 2 actions, later, when the pi matrix is customizable size, we can relatively easily make this matrix a customizable size
        # initialize line to change
        line = np.zeros(shape=state*action)
        # this double loop will go over every state-action combination
        for i in range(state):
            for k in range(action):
                # only works for player 1 for the moment
                value = pi_player_1[k][i][j]
                # make sure we negate 1 if i is equal to j so that we have the -c(j,k) that is on the right of equation 31 in the paper
                value -= i == j
                line[i*action + k] = value
                # The line is ordered so it does all the actions of one state before moving on to the next state, the vector of c values need to be ordered in the same way.
        # adds the line to the matrix to build it line by line
        A_matrix = np.vstack((A_matrix, line))
    return A_matrix

build_A_matrix()


array([[ 1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ],
       [-1.        , -1.        ,  0.2       ,  0.13333333,  0.        ,
         0.04      ,  0.27272727,  0.1025641 ,  0.1875    ,  0.27586207,
         0.46666667,  0.23529412],
       [ 0.47058824,  0.1       , -1.        , -1.        ,  0.7       ,
         0.2       ,  0.22727273,  0.15384615,  0.28125   ,  0.13793103,
         0.33333333,  0.11764706],
       [ 0.        ,  0.45      ,  0.36666667,  0.        , -1.        ,
        -1.        ,  0.18181818,  0.02564103,  0.25      ,  0.03448276,
         0.13333333,  0.35294118],
       [ 0.29411765,  0.        ,  0.2       ,  0.13333333,  0.        ,
         0.36      , -1.        , -1.        ,  0.09375   ,  0.24137931,
         0.        ,  0.        ],
       [ 0.23529412,  0.4       ,  0.03333333,  0.3       ,  0.1       ,
         0.36      ,  

As mentioned in the comments, this function assumes that the vector of c values is in the order such that all actions are done for one state before moving onto the next state. For example, we have c(i,k) such that i is the state and k is the action, the vector that is multiplied by A is in the order [c(1,1), c(1,2), c(2,1), c(2,2), c(3,1), c(3,2)... c(6,1), c(6,2)]. the result of the multiplication must be equal to b, where b = [1, 0, 0, 0, 0, 0, 0], a 1 followed by a 0 for every state, in this case 6 states.